# Impute the missing data
Treat the records without any data (no-admission) with masking


In [ ]:
import pandas as pd
import os
from pathlib import Path
import numpy as np

from data_preprocess.impute_fn import impute_ts_ehr_bffill, impute_ts_ehr_global, impute_ts_ehr_linear_interp

In [ ]:
# load data
time_resolution = "1h"
interested_split = 'test'
impute_method = "bffill" # choose from bffill, global, linearinterp

if time_resolution == "2h":
    num_timestep = 12
elif time_resolution == "1h":
    num_timestep = 24
elif time_resolution == "30min":
    num_timestep = 48
else:
    raise ValueError(f"Unknown time resolution {time_resolution}")

split_dir = f"data/eICU_first24h_ts{time_resolution}/split"
if impute_method == "bffill": # default impute method
    export_dir = f"data/eICU_first24h_ts{time_resolution}/imputed"
else:
    export_dir = f"data/eICU_first24h_ts{time_resolution}_impute-{impute_method}/imputed"

Path(os.path.join(export_dir, interested_split)).mkdir(exist_ok=True, parents=True)

train_ts_path = os.path.join(split_dir, 'train', 'time-series.csv')

demo_path = os.path.join(split_dir, interested_split, 'demographics.csv')
ts_path = os.path.join(split_dir, interested_split, 'time-series.csv')
label_icumortality_path = os.path.join(split_dir, interested_split, "label_icumortality.csv")
label_los_path = os.path.join(split_dir, interested_split, "label_los.csv")
label_medication_path = os.path.join(split_dir, interested_split, "label_medication.csv")
label_phenotype_path = os.path.join(split_dir, interested_split, "label_phenotype.csv")

demo_export_path = os.path.join(export_dir, interested_split, 'demographics.csv')
ts_export_path = os.path.join(export_dir, interested_split, 'time-series.csv')
label_icumortality_export_path = os.path.join(export_dir, interested_split, "label_icumortality.csv")
label_los_export_path = os.path.join(export_dir, interested_split, "label_los.csv")
label_medication_export_path = os.path.join(export_dir, interested_split, "label_medication.csv")
label_phenotype_export_path = os.path.join(export_dir, interested_split, "label_phenotype.csv")

train_ts_df = pd.read_csv(train_ts_path)

demo_df = pd.read_csv(demo_path)
ts_df = pd.read_csv(ts_path)
label_icumortality_df = pd.read_csv(label_icumortality_path)
label_los_df = pd.read_csv(label_los_path)
label_medication_df = pd.read_csv(label_medication_path)
label_phenotype_df = pd.read_csv(label_phenotype_path)

print(f"stay_id num of demo_df:{demo_df['stay_id'].unique().shape[0]}")
print(f"stay_id num of vital_df:{ts_df['stay_id'].unique().shape[0]}")
print(f"stay_id num of label_icumortality_df:{label_icumortality_df['stay_id'].unique().shape[0]}")
print(f"stay_id num of label_los_df:{label_los_df['stay_id'].unique().shape[0]}")
print(f"stay_id num of label_medication_df:{label_medication_df['stay_id'].unique().shape[0]}")
print(f"stay_id num of label_phenotype_df:{label_phenotype_df['stay_id'].unique().shape[0]}")

## Get the wanted features

First fill all the empty timestep with nan

In [ ]:
used_stay_id = demo_df['stay_id'].unique()
ts_df = ts_df[ts_df['stay_id'].isin(used_stay_id)]
label_icumortality_df = label_icumortality_df[label_icumortality_df['stay_id'].isin(used_stay_id)]
label_los_df = label_los_df[label_los_df['stay_id'].isin(used_stay_id)]
label_medication_df = label_medication_df[label_medication_df['stay_id'].isin(used_stay_id)]
label_phenotype_df = label_phenotype_df[label_phenotype_df['stay_id'].isin(used_stay_id)]

features_columns = [
    'heartrate_min', 'heartrate_max', 'heartrate_mean',
    'systemicsystolic_min', 'systemicsystolic_max', 'systemicsystolic_mean',
    'systemicdiastolic_min', 'systemicdiastolic_max', 'systemicdiastolic_mean',
    'systemicmean_min', 'systemicmean_max', 'systemicmean_mean',
    'respiration_min', 'respiration_max', 'respiration_mean',
    'temperature_min', 'temperature_max', 'temperature_mean',
    'sao2_min', 'sao2_max', 'sao2_mean',
    'glucose_min', 'glucose_max', 'glucose_mean',
    'HCO3', 'Hct', 'Hgb', 'PT', 'PTT', 'albumin', 'anion gap',
    'chloride', 'creatinine', 'lactate', 'platelets x 1000', 'sodium',
    'total bilirubin'
]

print(f"stay_id num of demo_df:{demo_df['stay_id'].unique().shape[0]}")
print(f"stay_id num of vital_df:{ts_df['stay_id'].unique().shape[0]}")
print(f"stay_id num of label_icumortality_df:{label_icumortality_df['stay_id'].unique().shape[0]}")
print(f"stay_id num of label_los_df:{label_los_df['stay_id'].unique().shape[0]}")
print(f"stay_id num of label_medication_df:{label_medication_df['stay_id'].unique().shape[0]}")
print(f"stay_id num of label_phenotype_df:{label_phenotype_df['stay_id'].unique().shape[0]}")


In [ ]:
# set the timepoints
unique_stay_ids = ts_df[["subject_id", "stay_id"]].drop_duplicates()
timepoints = pd.DataFrame({"timepoint": range(num_timestep)}) 

complete_timepoints = (
    unique_stay_ids.merge(timepoints, how="cross")
)

ts_df = complete_timepoints.merge(ts_df, on=["subject_id", "stay_id", "timepoint"], how="left")

print(ts_df["timepoint"].value_counts())

## Process missing value

In [ ]:
# impute missing value
id_col = "stay_id"
if impute_method == "bffill":
    ts_df, overall_dict = impute_ts_ehr_bffill(ts_df, features_columns, train_ts_df=train_ts_df, id_col=id_col)
elif impute_method == "linearinterp":
    ts_df, overall_dict = impute_ts_ehr_linear_interp(ts_df, features_columns, train_ts_df=train_ts_df, id_col=id_col)
elif impute_method == "global":
    ts_df, overall_dict = impute_ts_ehr_global(ts_df, features_columns, train_ts_df=train_ts_df, id_col=id_col)
else:
    raise ValueError(f"Unknown impute method {impute_method}")

## Process the ethnic and gender feature

In [ ]:
# Ethnicity before processing
print(demo_df["ethnicity"].unique())
print(demo_df["gender"].unique())

In [ ]:
def categorize_ethnicity(ethnicity):
    if pd.isna(ethnicity) or not isinstance(ethnicity, str):
        return "Unknown"
    
    if any(keyword in ethnicity for keyword in ["Caucasian"]):
        return "White"
    elif any(keyword in ethnicity for keyword in ["Asian"]):
        return "Asian"
    elif any(keyword in ethnicity for keyword in ["African American"]):
        return "Black"
    elif any(keyword in ethnicity for keyword in ['Hispanic']):
        return "Hispanic"
    else:
        return "Other"

def categorize_gender(gender):
    if pd.isna(gender) or not isinstance(gender, str):
        return "Unknown"
    
    if gender == "Male":
        return "Male"
    elif gender == "Female":
        return "Female"
    elif gender == "Other":
        return "Other"
    else:
        return "Unknown"

demo_df['ethnicity'] = demo_df['ethnicity'].apply(categorize_ethnicity)
demo_df['gender'] = demo_df["gender"].apply(categorize_gender)

print(demo_df["ethnicity"].unique())
print(demo_df["gender"].unique())

In [ ]:
ethnic_map = {"White": 0, "Black": 1, "Asian": 2, "Hispanic": 3, "Other": 4, "Unknown": 5}
demo_df["ethnicity_category"] = demo_df["ethnicity"].map(ethnic_map)

gender_map = {"Female": 0, "Male": 1, "Other": 2, "Unknown": 3}
demo_df["gender_category"] = demo_df["gender"].map(gender_map)
demo_df.drop(columns=["diagnosispriority", "seq_num"], inplace=True)

demo_df.head()

## Save data

In [ ]:
# save dfs
ts_df.to_csv(ts_export_path, index=False, sep=',')
demo_df.to_csv(demo_export_path, index=False, sep=',')
label_icumortality_df.to_csv(label_icumortality_export_path, index=False, sep=',')
label_los_df.to_csv(label_los_export_path, index=False, sep=',')
label_medication_df.to_csv(label_medication_export_path, index=False, sep=',')
label_phenotype_df.to_csv(label_phenotype_export_path, index=False, sep=',')
print(f"save to {export_dir} done")